# 02 — Resultados do braço A (só números)

Análises definidas no [pré-registro](../PREREGISTRATION.md), commitado antes de gerar qualquer previsão de teste.
As tabelas vêm de `uv run python -m supplyrisk.analise teste` (e `final`) e ficam em `data/resultados/<periodo>/`.

- **Teste:** 2015–2024 (10 anos). **Teste final:** 2025–2026, rodado uma única vez.
- **Tarefas principais:** 17,70 m em 14 e 30 dias. As de 15,0 m são secundárias (só 2 anos com evento no teste).
- **Métrica principal:** Brier score (menor é melhor). IC 95% por bootstrap reamostrando anos.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from supplyrisk import amostras as am
from supplyrisk import avaliacao as av
from supplyrisk.experimento import RESULTADOS

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

# Paleta categórica validada (ordem fixa): Jev, melhor baseline, climatologia, análogos
COR = {"jev": "#2a78d6", "baseline": "#eb6834", "climatologia": "#1baf7a", "analogos": "#eda100"}
TINTA, TINTA_2, GRADE = "#1f1f1e", "#5f5e58", "#e6e5df"
plt.rcParams.update({"axes.edgecolor": GRADE, "axes.labelcolor": TINTA_2, "xtick.color": TINTA_2,
                     "ytick.color": TINTA_2, "axes.titlecolor": TINTA, "axes.grid": True, "grid.color": GRADE,
                     "axes.spines.top": False, "axes.spines.right": False, "font.size": 10})

TESTE = RESULTADOS / "teste"
previsoes = pd.read_parquet(TESTE / "previsoes_com_calibracao.parquet")
principais = pd.read_csv(TESTE / "comparacoes_principais.csv")
robustez = pd.read_csv(TESTE / "robustez.csv")
memoria = pd.read_csv(TESTE / "memoria.csv")
antecedencia = pd.read_csv(TESTE / "antecedencia.csv")
alarmes = pd.read_csv(TESTE / "alarmes_falsos.csv")
MELHOR = {(1770, 14): "logistica__so_manaus", (1770, 30): "logistica__so_manaus",
          (1500, 14): "gradient_boosting__so_manaus", (1500, 30): "gradient_boosting__so_manaus"}

## 1. Comparações principais

1. **Jev bruto × climatologia:** o Jev tem habilidade sem nenhum treino?
2. **Jev calibrado × melhor baseline calibrado:** com o mesmo acesso à validação, o Jev compete?

`diferenca` = Brier do Jev − Brier do outro (negativo = Jev melhor). O veredito vem do IC 95%.

In [ ]:
principais[["tarefa", "tipo", "comparacao", "brier_jev", "brier_outro", "outro", "diferenca", "ic95", "veredito"]].round(4)

## 2. Brier por método, com IC 95%

Cada ponto é o Brier no teste, e a barra é o IC 95% (bootstrap por ano). Quanto mais à esquerda, melhor.

In [ ]:
linhas = []
for (limiar, h), g in previsoes.groupby(["limiar_cm", "horizonte_dias"]):
    for nome, coluna, cor in [
        ("Climatologia", "climatologia__principal", COR["climatologia"]),
        ("Análogos (só Manaus)", "analogos__so_manaus", COR["analogos"]),
        ("Melhor baseline, calibrado", MELHOR[(limiar, h)] + "__cal", COR["baseline"]),
        ("Jev bruto", "jev__principal", COR["jev"]),
        ("Jev calibrado", "jev__principal__cal", COR["jev"]),
    ]:
        lo, hi = av.bootstrap_por_ano(g, coluna)["brier_ic95"]
        linhas.append({"tarefa": f"{limiar / 100:.2f} m / {h} d", "metodo": nome, "cor": cor,
                       "brier": av.brier(g["rotulo"], g[coluna]), "lo": lo, "hi": hi})
tabela_brier = pd.DataFrame(linhas)

tarefas = ["17.70 m / 14 d", "17.70 m / 30 d", "15.00 m / 14 d", "15.00 m / 30 d"]
fig, eixos = plt.subplots(1, 4, figsize=(16, 3.6), sharey=True)
for ax, tarefa in zip(eixos, tarefas):
    t = tabela_brier[tabela_brier["tarefa"] == tarefa].reset_index(drop=True)
    for i, r in t.iterrows():
        ax.plot([r.lo, r.hi], [i, i], color=r.cor, lw=2, solid_capstyle="round")
        ax.plot(r.brier, i, "o", ms=8, color=r.cor, mec="white", mew=2,
                mfc="white" if r.metodo == "Jev bruto" else r.cor)
        ax.annotate(f"{r.brier:.3f}", (r.hi, i), xytext=(4, 0), textcoords="offset points",
                    va="center", fontsize=8, color=TINTA_2)
    ax.set_yticks(range(len(t)), t["metodo"])
    ax.set_title(tarefa + (" (principal)" if tarefa.startswith("17") else " (secundária)"), fontsize=10)
    ax.set_xlabel("Brier score (menor = melhor)")
    ax.invert_yaxis()
plt.tight_layout()
tabela_brier.drop(columns="cor").round(4)

## 3. Calibração: quando o método diz X%, acontece X% das vezes?

Tarefas principais. A diagonal é a calibração perfeita. Faixas com poucas amostras são menos confiáveis; o número de amostras de cada faixa está na tabela.

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(11, 4.6))
tabelas = {}
for ax, (limiar, h) in zip(eixos, [(1770, 14), (1770, 30)]):
    g = previsoes[(previsoes["limiar_cm"] == limiar) & (previsoes["horizonte_dias"] == h)]
    ax.plot([0, 1], [0, 1], color=GRADE, lw=1.5, ls="--")
    for nome, coluna, cor, vazio in [("Jev bruto", "jev__principal", COR["jev"], True),
                                     ("Jev calibrado", "jev__principal__cal", COR["jev"], False),
                                     ("Melhor baseline, calibrado", MELHOR[(limiar, h)] + "__cal", COR["baseline"], False)]:
        c = av.confiabilidade(g["rotulo"], g[coluna], bins=5)
        tabelas[(f"{limiar / 100:.2f} m / {h} d", nome)] = c
        ax.plot(c["prevista"], c["observada"], "-o", color=cor, lw=2, ms=8, mec="white", mew=2,
                mfc="white" if vazio else cor, label=nome)
    ax.set(xlim=(0, 1), ylim=(0, 1), xlabel="probabilidade prevista", ylabel="frequência observada",
           title=f"{limiar / 100:.2f} m em {h} dias")
    ax.legend(frameon=False, fontsize=8, loc="upper left")
plt.tight_layout()
pd.concat(tabelas, names=["tarefa", "metodo"]).round(3)

## 4. Um ano de perto: 2024 (seca recorde)

No painel de cima, a probabilidade de a cota chegar a 17,70 m em 30 dias, dia a dia. No de baixo, a cota real. São dois painéis separados para não pôr dois eixos y no mesmo gráfico.

In [ ]:
serie, _ = am.carregar_series()
ano, limiar, h = 2024, 1770, 30
g = previsoes[(previsoes["ano"] == ano) & (previsoes["limiar_cm"] == limiar) & (previsoes["horizonte_dias"] == h)].sort_values("data")
fig, (a1, a2) = plt.subplots(2, 1, figsize=(11, 6), sharex=True, gridspec_kw={"height_ratios": [1.2, 1]})
for nome, coluna, cor in [("Jev bruto", "jev__principal", COR["jev"]),
                          ("Melhor baseline, calibrado", MELHOR[(limiar, h)] + "__cal", COR["baseline"]),
                          ("Climatologia", "climatologia__principal", COR["climatologia"])]:
    a1.plot(g["data"], g[coluna], color=cor, lw=2, label=nome)
a1.axhline(0.5, color=TINTA_2, lw=1, ls=":")
a1.set(ylim=(0, 1), ylabel="P(chegar a 17,70 m em 30 d)", title=f"{ano}: previsões dia a dia")
a1.legend(frameon=False, fontsize=8, loc="upper left")
trecho = serie[f"{ano}-07-01":f"{ano}-11-30"] / 100
a2.plot(trecho.index, trecho, color=TINTA, lw=2)
a2.axhline(limiar / 100, color=TINTA_2, lw=1, ls="--")
a2.annotate("17,70 m", (trecho.index[0], limiar / 100), xytext=(0, 4), textcoords="offset points", fontsize=8, color=TINTA_2)
a2.set(ylabel="cota em Manaus (m)")
plt.tight_layout()

## 5. Robustez do Jev

Mudanças que não deveriam alterar a resposta: o nome das opções, a ordem das opções e o tipo de pergunta. `decisoes_invertidas` é a fração de dias em que a decisão (P ≥ 0,5) muda em relação à versão principal.

In [ ]:
robustez.assign(tarefa=lambda d: d["limiar_cm"].div(100).map("{:.2f} m".format) + " / " + d["horizonte_dias"].astype(str) + " d")[
    ["tarefa", "variante", "brier_principal", "brier", "dif_abs_media", "decisoes_invertidas"]].round(4)

## 6. Teste de memória

A mesma pergunta, mas com a data e o ano no estado. Se o Jev lembrasse das secas conhecidas, o Brier **com datas** cairia nos anos de seca, o que apareceria como valores negativos na tabela.

In [ ]:
memoria.pivot_table(index=["ano", "cruzou"], columns=["limiar_cm", "horizonte_dias"], values="com_menos_sem").round(4)

## 7. Antecedência e alarmes falsos

- **Antecedência:** quantos dias seguidos o método ficou com P ≥ 0,5 antes do primeiro cruzamento.
- **Alarme falso:** um ano sem cruzamento em que o método passou de 0,5 em algum dia.

In [ ]:
antecedencia.pivot_table(index=["limiar_cm", "horizonte_dias", "ano"], columns="metodo", values="dias_de_alerta")

In [ ]:
alarmes.pivot_table(index=["limiar_cm", "horizonte_dias", "anos_sem_evento"], columns="metodo", values="anos_com_alarme_falso")

## 8. Teste final (2025–2026)

Rodado uma única vez, depois de tudo acima. Como declarado no pré-registro, não há cruzamentos com rótulo conhecido nesse período, então essa parte mede só alarmes falsos e calibração em dias sem evento.

In [ ]:
FINAL = RESULTADOS / "final"
if (FINAL / "metricas.csv").exists():
    final = pd.read_csv(FINAL / "metricas.csv")
    destaque = ["climatologia__principal", "logistica__so_manaus__cal", "gradient_boosting__so_manaus__cal",
                "jev__principal", "jev__principal__cal"]
    display(final[final["metodo"].isin(destaque)][["limiar_cm", "horizonte_dias", "metodo", "n", "positivos", "brier"]].round(4))
    display(pd.read_csv(FINAL / "alarmes_falsos.csv").pivot_table(index=["limiar_cm", "horizonte_dias"], columns="metodo", values="anos_com_alarme_falso"))
else:
    print("Teste final ainda não rodado.")